In [1]:
import polars as pl
import polars.selectors as cs
import pandas as pd
import numpy as np
import glob
import os
import io
from hta.trace_analysis import TraceAnalysis

In [9]:
analyzer = TraceAnalysis({0: "device.0.json"}, "/proj/threadtune-PG0/amir/kineto_traces/bert-inference")
rank = 0
trace_data = analyzer.t.get_trace(rank)
sym = analyzer.t.symbol_table.get_sym_table()

Parsed /proj/threadtune-PG0/amir/kineto_traces/bert-inference/device.0.json time = 0.02 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425


Parsed /proj/threadtune-PG0/amir/kineto_traces/bert-inference/device.0.json backend=ParserBackend.JSON in 0.07 seconds; current PID:231144
Overall parsing of /proj/threadtune-PG0/amir/kineto_traces/bert-inference/device.0.json in 0.09 seconds; current PID:231144
leaving parse_multiple_ranks duration=0.09 seconds
leaving parse_traces duration=0.09 seconds
There is only one iteration in the trace. The analysis result may not be accurate.


In [15]:
df = (
    pl.from_pandas(trace_data[["index", "ts", "dur", "cat", "name"]])
    .with_columns(
        pl.col(col).map_elements(lambda x: sym[x]) for col in ["cat", "name"]
    )
)

In [29]:
df["cat"].unique()

cat
str
"""gpu_user_annotation"""
"""cuda_sync"""
"""kernel"""
"""gpu_memcpy"""
"""cuda_runtime"""
"""user_annotation"""
"""overhead"""
"""cpu_op"""


In [31]:
kernel_df = df.filter(pl.col("cat") == "kernel")

In [37]:
(
    kernel_df.join(kernel_df, how="cross")
    .filter(
        (pl.col("index") != pl.col("index_right")) &
        pl.col("ts").is_between(pl.col("ts_right"), pl.col("ts_right") + pl.col("dur_right")) &
        (pl.col("ts") + pl.col("dur")).is_between(pl.col("ts_right"), pl.col("ts_right") + pl.col("dur_right"))
    )
    .is_empty()
)

True

In [34]:
kernel_df["dur"].sum()

1601.0

In [18]:
cpu_df = df.filter(pl.col("cat") == "cpu_op")

In [21]:
cpu_roots_df = cpu_df.join(
    cpu_df.join(cpu_df, how="cross")
    .filter(
        (pl.col("ts") > pl.col("ts_right")) &
        (pl.col("ts") + pl.col("dur") < pl.col("ts_right") + pl.col("dur_right"))
    ),
    on="index",
    how="anti"
)

In [38]:
(
    cpu_roots_df.join(cpu_roots_df, how="cross")
    .filter(
        (pl.col("index") != pl.col("index_right")) &
        pl.col("ts").is_between(pl.col("ts_right"), pl.col("ts_right") + pl.col("dur_right")) &
        (pl.col("ts") + pl.col("dur")).is_between(pl.col("ts_right"), pl.col("ts_right") + pl.col("dur_right"))
    )
    .is_empty()
)

True

In [22]:
cpu_roots_df["dur"].sum()

204052.0

In [42]:
kernel_df.join(
    kernel_df.join(cpu_roots_df, how="cross")
    .filter(
        pl.col("ts").is_between(pl.col("ts_right"), pl.col("ts_right") + pl.col("dur_right")) &
        (pl.col("ts") + pl.col("dur")).is_between(pl.col("ts_right"), pl.col("ts_right") + pl.col("dur_right"))
    ),
    on="index",
    how="anti"
).is_empty()

True

In [11]:
(
    pl.concat(
        pl.from_pandas(trace_data[trace_data["stream"] != -1])
        .select(pl.lit(rank).alias("rank"), pl.all())
        for rank, trace_data in analyzer.t.traces.items()
    ).join(
        pl.from_dict({"name": list(range(len(sym))), "s_name": sym}),
        on="name",
    ).with_columns(
        s_cat=pl.col("cat").map_elements(lambda x: sym[x])
    ).select(
        "index", "ts", "dur", "cat", "name", "s_cat", "s_name"
    )
)

index,ts,dur,cat,name,s_cat,s_name
i16,i64,f64,i64,i64,str,str
1896,160569,9.0,69,16,"""kernel""","""void at::native::(anonymous na…"
1900,161062,4.0,69,16,"""kernel""","""void at::native::(anonymous na…"
1907,172665,1.0,69,2,"""kernel""","""void at::native::vectorized_el…"
1911,172961,7.0,69,16,"""kernel""","""void at::native::(anonymous na…"
1915,173103,0.0,69,2,"""kernel""","""void at::native::vectorized_el…"
…,…,…,…,…,…,…
2957,311588,7.0,69,56,"""kernel""","""void gemv2T_kernel_val<int, in…"
2964,313198,2.0,69,45,"""kernel""","""void at::native::vectorized_el…"
2972,313485,3.0,69,27,"""kernel""","""void at::native::(anonymous na…"


In [2]:
analyzer = TraceAnalysis({0: "device-1100.0.json"}, "/proj/threadtune-PG0/amir/V100_NEW_DATA/bert-inference/")
sym = analyzer.t.symbol_table.get_sym_table()

(pl.concat([
    pl.from_pandas(analyzer.t.get_trace(r)[analyzer.t.get_trace(r)["stream"] != -1][["name", "dur"]])
    for r in analyzer.t.traces
]).join(
    pl.from_dict({"name": list(range(len(sym))), "s_name": sym}), on="name"
).filter(
    ~pl.col("s_name").str.starts_with("nccl") & 
    ~pl.col("s_name").is_in(["Stream Sync", "Memcpy DtoD (Device -> Device)", "Memcpy HtoD (Pageable -> Device)", "Memcpy DtoH (Device -> Pinned)"])
).select(
    pl.lit("bert-inference").alias("benchmark"),
    pl.col("dur").sum().alias("measured_kernel_time_sum_us")
))

Parsed /proj/threadtune-PG0/amir/V100_NEW_DATA/bert-inference/device-1100.0.json time = 0.26 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425
Parsed /proj/threadtune-PG0/amir/V100_NEW_DATA/bert-inference/device-1100.0.json backend=ParserBackend.JSON in 1.03 seconds; current PID:185653
Overall parsing of /proj/threadtune-PG0/amir/V100_NEW_DATA/bert-inference/device-1100.0.json in 1.18 seconds; current PID:185653
leaving parse_multiple_ranks duration=1.21 seconds
leaving parse_traces duration=1.21 seconds
There is only one iteration in the trace. The analysis result may not be accurate.


benchmark,measured_kernel_time_sum_us
str,f64
"""bert-inference""",16337.0


In [3]:
sym[546]

'nn.Module: LayerNorm_15'

In [4]:
dir(analyzer.t)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_align_all_ranks',
 '_filter_irrelevant_gpu_kernels',
 '_get_first_rank',
 '_normalize_trace_filenames',
 '_validate_trace_files',
 'align_and_filter_trace',
 'convert_time_series_to_events',
 'decode_symbol_ids',
 'flow_event',
 'get_all_traces',
 'get_iterations',
 'get_ranks',
 'get_raw_trace_for_one_rank',
 'get_trace',
 'get_trace_duration',
 'get_trace_start_unixtime_ns',
 'is_parsed',
 'load_traces',
 'meta_data',
 'min_ts',
 'parse_multiple_ranks',
 'parse_single_rank',
 'parse_traces',
 'parser_config',
 'symbol_table',
 'trace_files',
 'trace_path',
 'traces',
 'write_raw_trace']

In [5]:
rank = 0
trace_data = analyzer.t.get_trace(rank)
symbol_table = analyzer.t.symbol_table.get_sym_table()
df = (
    pl.concat(
        pl.from_pandas(trace_data[trace_data["stream"] != -1])
        .select(pl.lit(rank).alias("rank"), pl.all())
        for rank, trace_data in analyzer.t.traces.items()
    ).join(
        pl.from_dict({"name": list(range(len(symbol_table))), "s_name": symbol_table}),
        on="name",
    ).with_columns(
        s_cat=pl.col("cat").map_elements(lambda x: sym[x])
    )
)

In [6]:
df

rank,index,cat,name,pid,tid,ts,dur,end,stream,correlation,bytes,memory_bw_gbps,wait_on_stream,wait_on_cuda_event_record_corr_id,input_dims,input_type,input_strides,external_id,index_correlation,iteration,s_name,s_cat
i32,i32,i64,i64,i64,i64,i64,f64,f64,i8,i32,i8,f64,i8,i8,str,str,str,i32,i32,i8,str,str
0,50410,126,257,0,7,231831,8.0,4.2494e12,7,22,-1,0.0,-1,-1,"""-1""","""-1""","""-1""",12,50412,0,"""void at::native::(anonymous na…","""kernel"""
0,50414,126,257,0,7,232364,4.0,4.2494e12,7,41,-1,0.0,-1,-1,"""-1""","""-1""","""-1""",19,50416,0,"""void at::native::(anonymous na…","""kernel"""
0,50420,126,156,0,7,247249,1.0,4.2494e12,7,51,-1,0.0,-1,-1,"""-1""","""-1""","""-1""",23,50422,0,"""void at::native::vectorized_el…","""kernel"""
0,50424,126,257,0,7,247567,7.0,4.2494e12,7,70,-1,0.0,-1,-1,"""-1""","""-1""","""-1""",27,50426,0,"""void at::native::(anonymous na…","""kernel"""
0,50428,126,156,0,7,247709,1.0,4.2494e12,7,76,-1,0.0,-1,-1,"""-1""","""-1""","""-1""",31,50430,0,"""void at::native::vectorized_el…","""kernel"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0,62138,433,443,0,7,1516231,2.0,4.2494e12,7,45141,-1,0.0,-1,-1,"""-1""","""-1""","""-1""",14226,62140,0,"""Stream Sync""","""cuda_sync"""
0,62142,323,536,0,7,1516329,1.0,4.2494e12,7,45146,1,0.000679,-1,-1,"""-1""","""-1""","""-1""",14229,62144,0,"""Memcpy DtoH (Device -> Pinned)""","""gpu_memcpy"""
0,62146,433,443,0,7,1516329,7.0,4.2494e12,7,45147,-1,0.0,-1,-1,"""-1""","""-1""","""-1""",14229,62148,0,"""Stream Sync""","""cuda_sync"""


In [7]:
pl.read_json("/proj/threadtune-PG0/amir/V100_NEW_DATA/bert-inference/device-1100.0.json").select(pl.col("traceEvents").explode().struct.unnest())

ph,cat,ts,dur,pid,tid,name,args,s,id,bp
str,str,f64,f64,str,str,str,struct[37],str,i64,str
"""X""","""user_annotation""",4.2494e12,1.5163e6,"""147641""","""147641""","""ProfilerStep#0""","{1,null,null,null,null,null,null,null,0,null,null,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null}",null,null,null
"""X""","""user_annotation""",4.2494e12,1.5161e6,"""147641""","""147641""","""bert-inference""","{2,null,null,null,null,null,null,null,1,null,null,2,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null}",null,null,null
"""X""","""cpu_op""",4.2494e12,165.2,"""147641""","""147641""","""aten::slice""","{3,null,null,null,null,null,null,null,2,null,null,3,["""", ""1"", … ""1""],[""long int"", ""Scalar"", … ""Scalar""],[[512, 1], [], … []],[[1, 512], [], … []],null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null}",null,null,null
"""X""","""cpu_op""",4.2494e12,59.604,"""147641""","""147641""","""aten::as_strided""","{4,null,null,null,null,null,null,null,3,null,null,4,["""", ""[1, 8]"", … ""0""],[""long int"", ""ScalarList"", … ""Scalar""],[[512, 1], [], … []],[[1, 512], [], … []],null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null}",null,null,null
"""X""","""cpu_op""",4.2494e12,79.319,"""147641""","""147641""","""aten::expand""","{5,null,null,null,null,null,null,null,4,null,null,5,["""", ""[1, 8]"", ""False""],[""long int"", ""ScalarList"", ""Scalar""],[[512, 1], [], []],[[1, 8], [], []],null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null}",null,null,null
…,…,…,…,…,…,…,…,…,…,…
"""M""",null,4.2494e12,null,"""147641""","""147641""","""thread_sort_index""","{null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,147641,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null}",null,null,null
"""X""","""Trace""",4.2494e12,1.5177e6,"""Spans""","""PyTorch Profiler""","""PyTorch Profiler (0)""","{null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null}",null,null,null
"""M""",null,4.2494e12,null,"""Spans""","""0""","""process_sort_index""","{null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,536870912,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null}",null,null,null
